# Week 01 · Day 03 — Vector Databases

**Topics:** ChromaDB operations · Metadata filtering · Pinecone · Qdrant · Hybrid search with RRF


In [ ]:
%pip install -q openai chromadb

In [ ]:
import os
from openai import OpenAI
import chromadb

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
chroma = chromadb.Client()

def embed(texts: list[str]) -> list[list[float]]:
    r = client.embeddings.create(input=texts, model="text-embedding-3-small")
    return [item.embedding for item in r.data]

## 1 · ChromaDB — CRUD Operations

In [ ]:
# Create a collection
col = chroma.get_or_create_collection("products")

products = [
    {"id": "p001", "text": "Wireless noise-cancelling headphones with 30-hour battery", "category": "audio", "price": 299, "in_stock": True},
    {"id": "p002", "text": "Mechanical keyboard with RGB backlight and Cherry MX switches", "category": "peripherals", "price": 149, "in_stock": True},
    {"id": "p003", "text": "4K webcam with built-in ring light for video calls", "category": "video", "price": 89, "in_stock": False},
    {"id": "p004", "text": "Over-ear studio monitor headphones for music production", "category": "audio", "price": 199, "in_stock": True},
    {"id": "p005", "text": "Ergonomic wireless mouse with vertical grip design", "category": "peripherals", "price": 79, "in_stock": True},
    {"id": "p006", "text": "Bluetooth portable speaker with 360-degree sound", "category": "audio", "price": 59, "in_stock": True},
]

texts = [p["text"] for p in products]
embeddings = embed(texts)

col.add(
    documents=texts,
    embeddings=embeddings,
    ids=[p["id"] for p in products],
    metadatas=[{"category": p["category"], "price": p["price"], "in_stock": p["in_stock"]} for p in products],
)

print(f"Collection '{col.name}': {col.count()} documents")

In [ ]:
# Basic semantic query
query = "something for listening to music"
q_emb = embed([query])[0]

results = col.query(query_embeddings=[q_emb], n_results=3)
print(f"Query: '{query}'")
for doc, dist in zip(results["documents"][0], results["distances"][0]):
    print(f"  dist={dist:.4f} — {doc}")

## 2 · Metadata Filtering

ChromaDB supports `where` filters using `$eq`, `$ne`, `$gt`, `$lt`, `$in`, `$and`, `$or`.

In [ ]:
# Filter: only audio products that are in stock
results = col.query(
    query_embeddings=[embed(["wireless audio device"])[0]],
    n_results=5,
    where={
        "$and": [
            {"category": {"$eq": "audio"}},
            {"in_stock": {"$eq": True}},
        ]
    },
)

print("Audio products in stock:")
for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
    print(f"  ${meta['price']:3d} — {doc[:60]}")

In [ ]:
# Filter: products under $100
results = col.query(
    query_embeddings=[embed(["budget accessories"])[0]],
    n_results=5,
    where={"price": {"$lt": 100}},
)

print("Products under $100:")
for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
    print(f"  ${meta['price']:3d} — {doc[:60]}")

## 3 · Upsert and Delete

In [ ]:
# Update a document (upsert replaces if ID exists)
updated_text = "Wireless noise-cancelling headphones with 40-hour battery (updated)"
col.upsert(
    ids=["p001"],
    documents=[updated_text],
    embeddings=embed([updated_text]),
    metadatas=[{"category": "audio", "price": 279, "in_stock": True}],
)

# Verify update
item = col.get(ids=["p001"])
print("Updated p001:")
print(f"  Text:  {item['documents'][0]}")
print(f"  Price: ${item['metadatas'][0]['price']}")

# Delete a document
col.delete(ids=["p003"])
print(f"\nAfter deleting p003: {col.count()} documents remain")

## 4 · Hybrid Search (Dense + Sparse) with RRF

Dense search (embeddings) finds semantically similar docs but can miss exact terms. Sparse search (BM25) finds exact keyword matches but misses paraphrases. **Hybrid = best of both**.

**Reciprocal Rank Fusion (RRF)** merges ranked lists: `score = Σ 1 / (k + rank)` where k=60.

In [ ]:
from collections import defaultdict

def bm25_search(query: str, documents: list[str], top_k: int = 5) -> list[tuple[int, float]]:
    """Simplified BM25 — keyword overlap scoring."""
    query_terms = set(query.lower().split())
    scores = []
    for i, doc in enumerate(documents):
        doc_terms = doc.lower().split()
        overlap = sum(1 for t in doc_terms if t in query_terms)
        scores.append((i, overlap))
    return sorted(scores, key=lambda x: x[1], reverse=True)[:top_k]

def dense_search(query: str, docs_with_ids: list[dict], top_k: int = 5) -> list[tuple[str, float]]:
    q_emb = embed([query])[0]
    results = col.query(query_embeddings=[q_emb], n_results=top_k)
    return list(zip(results["ids"][0], results["distances"][0]))

def rrf_merge(ranked_lists: list[list], k: int = 60) -> list:
    scores = defaultdict(float)
    for ranked in ranked_lists:
        for rank, item in enumerate(ranked):
            scores[item] += 1.0 / (k + rank + 1)
    return sorted(scores.keys(), key=lambda x: scores[x], reverse=True)

# Hybrid search demo
query = "Cherry MX mechanical"
docs_text = [p["text"] for p in products]

# Dense results (by index)
dense_results = dense_search(query, products)
dense_ids = [id_ for id_, _ in dense_results]

# Sparse results (by index)
sparse_results = bm25_search(query, docs_text)
sparse_ids = [products[i]["id"] for i, _ in sparse_results]

# Merge with RRF
merged = rrf_merge([dense_ids, sparse_ids])

print(f"Hybrid search for: '{query}'")
for pid in merged[:3]:
    p = next((p for p in products if p["id"] == pid), None)
    if p:
        print(f"  {pid}: {p['text'][:60]}")

## 5 · Vector Database Comparison

| Feature | ChromaDB | Pinecone | Qdrant |
|---------|----------|----------|--------|
| Hosting | Self-hosted / embedded | Managed cloud | Self-hosted / cloud |
| Best for | Development, small scale | Production, multi-region | Payload filtering, hybrid |
| Metadata filter | `where=` dict | `filter=` dict | `Filter` object |
| Hybrid search | Manual (bring your own) | Built-in (sparse+dense) | Built-in |
| Free tier | Yes (local) | Starter plan | Yes (self-hosted) |

In [ ]:
# Pinecone pattern (requires: pip install pinecone-client)
# from pinecone import Pinecone, ServerlessSpec
# pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
# index = pc.Index("my-index")
# index.upsert(vectors=[("id1", embedding, {"category": "audio"})], namespace="products")
# results = index.query(vector=query_emb, top_k=5, filter={"category": {"$eq": "audio"}})

print("Pinecone pattern shown as comments — add PINECONE_API_KEY to run")

# Qdrant pattern (requires: pip install qdrant-client)
# from qdrant_client import QdrantClient
# from qdrant_client.models import Filter, FieldCondition, MatchValue
# qd = QdrantClient(url=os.getenv("QDRANT_URL"), api_key=os.getenv("QDRANT_API_KEY"))
# results = qd.search(
#     collection_name="products",
#     query_vector=query_emb,
#     query_filter=Filter(must=[FieldCondition(key="category", match=MatchValue(value="audio"))]),
#     limit=5,
# )

print("Qdrant pattern shown as comments — add QDRANT_URL + QDRANT_API_KEY to run")

## Summary

- ChromaDB: excellent for dev; switch to Pinecone or Qdrant for production scale.
- Metadata filters narrow the search space before embedding similarity is computed.
- Always `upsert` (not insert) to handle document updates without duplicate IDs.
- Hybrid search (dense + BM25 + RRF) outperforms pure dense search when queries contain exact product names or codes.

**Next:** [Evaluation](week01-day04-evaluation.ipynb)